---
## Stage 8 v2: Pair Construction & Label Building

**วัตถุประสงค์:** สร้าง labeled pairs (positive + negative + hard negative) สำหรับ training

**Input:** `all_profiles_cleaned.csv`, `candidate_pairs.csv` (จาก Stage 7 v2)  
**Output:** `labeled_pairs.csv`

### ปรับปรุงจาก v1
| ประเด็น | v1 (ผิด) | v2 (ถูก) |
|---------|----------|----------|
| Positive pair logic | groupby `profile_id` (ต่างแพลตฟอร์มมี profile_id ต่างกัน) | groupby `user_folder` (entity ID จริง) |
| ผล | positive pairs = 0 หรือน้อยมาก | positive pairs ครบถ้วนตาม ground truth |
| candidate_pairs | 70,271 pairs (recall 56.1%) | 331,586+ pairs (recall 84.1%) |

**Bug fix:** v1 ใช้ `profile_id` ซึ่งมีรูปแบบ `twitter_aaronbird` / `instagram_aaronbird` — เป็น string ต่างกัน  
v2 ใช้ `user_folder` เป็น entity ID จริง (คนเดียวกัน = `user_folder` เดียวกัน)

| Sub-step | หน้าที่ |
|----------|--------|
| 8.1 | Positive Pair Generation (user_folder groupby) |
| 8.2 | Random Negative Generation |
| 8.3 | Hard Negative Generation (from candidate_pairs) |
| 8.4 | Combine & Shuffle |

In [1]:
# ─── Config ───────────────────────────────────────────────────────────────
OUTPUT_DIR      = '/Users/tm/Documents/GitHub/Project-for-Work/data/processed'
PROFILES_CSV    = f'{OUTPUT_DIR}/all_profiles_cleaned.csv'
CANDIDATES_CSV  = f'{OUTPUT_DIR}/candidate_pairs.csv'
LABELED_CSV     = f'{OUTPUT_DIR}/labeled_pairs.csv'
RANDOM_SEED     = 42
NEG_TO_POS_RATIO = 5  # random negatives per positive
# ──────────────────────────────────────────────────────────────────────────

import os, itertools
import numpy as np
import pandas as pd

df_clean        = pd.read_csv(PROFILES_CSV)
candidate_pairs = pd.read_csv(CANDIDATES_CSV)
print(f'df_clean: {len(df_clean):,} | candidate_pairs: {len(candidate_pairs):,}')
print(f'Columns: {list(df_clean.columns)}')
assert 'user_folder' in df_clean.columns, 'ERROR: user_folder column missing from df_clean!'
print(f'user_folder unique: {df_clean["user_folder"].nunique():,}')

df_clean: 36,807 | candidate_pairs: 848,902
Columns: ['userName', 'fullName', 'bio', 'location', 'externalUrl', 'pictureURL', 'bio_urls', 'bio_url_count', 'bio_mentions', 'bio_mentions_count', 'location_type', 'location_valid', 'latitude', 'longitude', 'url_count', 'externalUrl_clean', 'external_domain', 'user_folder', 'platform', 'source_folder', 'bigrams', 'outputProfileName', 'userName_clean', 'fullName_clean', 'bio_clean', 'location_clean', 'url_domain', 'profile_id']
user_folder unique: 15,297


### Step 8.1: Positive Pair Generation (v2 — user_folder groupby)

**v2 fix:** groupby `user_folder` แทน `profile_id`  
- `user_folder` = entity ID จริง (คนเดียวกัน = user_folder เดียวกัน)  
- บังคับ cross-platform เท่านั้น (ไม่นับ pair ใน platform เดียวกัน)

In [2]:
# --- 8.1 Positive Pair Generation (v2: user_folder groupby) ---

def build_positive_pairs(df: pd.DataFrame) -> pd.DataFrame:
    """
    v2: สร้าง positive pairs โดย groupby user_folder
    user_folder เดียวกัน + platform ต่างกัน = คนเดียวกัน (ground truth)
    """
    pairs = []
    for folder, group in df.groupby('user_folder'):
        # ต้องมีอย่างน้อย 2 platform
        if group['platform'].nunique() < 2:
            continue
        for (i, row_a), (j, row_b) in itertools.combinations(group.iterrows(), 2):
            # cross-platform เท่านั้น
            if row_a['platform'] == row_b['platform']:
                continue
            pairs.append({
                'profile_id_a': row_a['profile_id'],
                'profile_id_b': row_b['profile_id'],
                'entity_id_a':  folder,
                'platform_a':   row_a['platform'],
                'platform_b':   row_b['platform'],
                'label':        1
            })
    return pd.DataFrame(pairs)

positive_pairs = build_positive_pairs(df_clean)

print('📊 Step 8.1: Positive Pair Generation (v2)')
print('=' * 60)
print(f'  Positive pairs   : {len(positive_pairs):,}')
if len(positive_pairs) > 0:
    print(f'  Unique entities  : {positive_pairs["entity_id_a"].nunique():,}')
    print(f'\n  Platform combo distribution:')
    combo = positive_pairs.apply(lambda r: f"{r['platform_a']}↔{r['platform_b']}", axis=1)
    print(combo.value_counts().to_string(header=False))
    print(f'\n  Sample positive pairs (5):')
    for _, r in positive_pairs.head(5).iterrows():
        print(f'    [{r["platform_a"]}] {str(r["profile_id_a"])[:30]} '
              f'↔ [{r["platform_b"]}] {str(r["profile_id_b"])[:30]}  (folder={r["entity_id_a"]})')
print(f'\n✅ Step 8.1 เสร็จ')

📊 Step 8.1: Positive Pair Generation (v2)
  Positive pairs   : 29,243
  Unique entities  : 13,771

  Platform combo distribution:
googleplus↔twitter      10710
instagram↔twitter       10183
googleplus↔instagram     8350

  Sample positive pairs (5):
    [googleplus] googleplus_caseycolettebyrddav ↔ [twitter] twitter_1caseycolette  (folder=1caseycolette)
    [googleplus] googleplus_rodneyhargis ↔ [twitter] twitter_rhargismccc  (folder=3twenty6)
    [googleplus] googleplus_mattcollinge ↔ [instagram] instagram_604homes  (folder=604homes)
    [googleplus] googleplus_mattcollinge ↔ [twitter] twitter_604homesguy  (folder=604homes)
    [instagram] instagram_604homes ↔ [twitter] twitter_604homesguy  (folder=604homes)

✅ Step 8.1 เสร็จ


### Step 8.2: Random Negative Pair Generation
สร้าง negative pairs (คนละ `user_folder`) ด้วย random sampling

In [3]:
# --- 8.2 Random Negative Pair Generation ---

def build_random_negatives(df: pd.DataFrame, n_neg: int, seed: int = 42) -> pd.DataFrame:
    """สร้าง random negative pairs (คนละ user_folder ข้าม platform)"""
    rng = np.random.default_rng(seed)
    pairs = []
    platforms = df['platform'].unique()
    platform_dfs = {p: df[df['platform'] == p].reset_index(drop=True) for p in platforms}
    platform_combos = list(itertools.combinations(platforms, 2))

    attempts = 0
    max_attempts = n_neg * 10

    while len(pairs) < n_neg and attempts < max_attempts:
        attempts += 1
        p1, p2 = platform_combos[rng.integers(len(platform_combos))]
        df1, df2 = platform_dfs[p1], platform_dfs[p2]
        row_a = df1.iloc[rng.integers(len(df1))]
        row_b = df2.iloc[rng.integers(len(df2))]
        # v2: ใช้ user_folder เพื่อตรวจสอบว่าคนละคน
        if str(row_a['user_folder']) != str(row_b['user_folder']):
            pairs.append({
                'profile_id_a': row_a['profile_id'],
                'profile_id_b': row_b['profile_id'],
                'entity_id_a':  row_a['user_folder'],
                'platform_a':   p1,
                'platform_b':   p2,
                'label':        0
            })
    return pd.DataFrame(pairs)

n_neg_random = len(positive_pairs) * NEG_TO_POS_RATIO
random_negatives = build_random_negatives(df_clean, n_neg_random, seed=RANDOM_SEED)

print('📊 Step 8.2: Random Negative Generation')
print('=' * 60)
print(f'  Target negatives : {n_neg_random:,} (ratio {NEG_TO_POS_RATIO}:1)')
print(f'  Generated        : {len(random_negatives):,}')
print(f'\n✅ Step 8.2 เสร็จ')

📊 Step 8.2: Random Negative Generation
  Target negatives : 146,215 (ratio 5:1)
  Generated        : 146,215

✅ Step 8.2 เสร็จ


### Step 8.3: Hard Negative Generation
Hard negatives = pairs จาก candidate_pairs (blocking output) ที่คล้ายกันแต่ **คนละ user_folder**  
→ ช่วย model เรียนรู้กรณีที่ยากขึ้น (คนละคนแต่ username/fullname คล้ายกัน)

In [4]:
# --- 8.3 Hard Negative Generation ---

def build_hard_negatives(df: pd.DataFrame, candidate_pairs: pd.DataFrame,
                         n_hard: int, seed: int = 42) -> pd.DataFrame:
    """
    Hard negatives จาก blocking candidates:
    - cross-platform
    - คนละ user_folder = negative
    """
    if candidate_pairs is None or len(candidate_pairs) == 0:
        return pd.DataFrame()

    # v2: lookup โดย profile_id → user_folder
    pid_to_folder = df.set_index('profile_id')['user_folder'].to_dict()
    pid_to_platform = df.set_index('profile_id')['platform'].to_dict()

    hard_pairs = []
    for _, row in candidate_pairs.iterrows():
        id_a = row['profile_id_a']
        id_b = row['profile_id_b']
        folder_a = pid_to_folder.get(id_a, '')
        folder_b = pid_to_folder.get(id_b, '')
        plat_a   = pid_to_platform.get(id_a, '')
        plat_b   = pid_to_platform.get(id_b, '')

        # cross-platform + คนละ user_folder = hard negative
        if plat_a != plat_b and str(folder_a) != str(folder_b) and folder_a and folder_b:
            hard_pairs.append({
                'profile_id_a': id_a,
                'profile_id_b': id_b,
                'entity_id_a':  folder_a,
                'platform_a':   plat_a,
                'platform_b':   plat_b,
                'label':        0
            })

    hard_df = pd.DataFrame(hard_pairs)
    if len(hard_df) > n_hard:
        hard_df = hard_df.sample(n=n_hard, random_state=seed)
    return hard_df

n_hard = len(positive_pairs)  # hard negatives = จำนวน positive
hard_negatives = build_hard_negatives(df_clean, candidate_pairs, n_hard, seed=RANDOM_SEED)

print('📊 Step 8.3: Hard Negative Generation')
print('=' * 60)
print(f'  Hard negatives available : {len(hard_negatives):,}')
print(f'  Target                   : {n_hard:,}')
print(f'  (from {len(candidate_pairs):,} candidate pairs)')
print(f'\n✅ Step 8.3 เสร็จ')

📊 Step 8.3: Hard Negative Generation
  Hard negatives available : 29,243
  Target                   : 29,243
  (from 848,902 candidate pairs)

✅ Step 8.3 เสร็จ


### Step 8.4: Combine & Shuffle All Pairs
รวม positive + random negative + hard negative → shuffle → save

In [5]:
# --- 8.4 Combine & Shuffle ---

all_parts = [positive_pairs]
if len(random_negatives) > 0:
    all_parts.append(random_negatives)
if len(hard_negatives) > 0:
    all_parts.append(hard_negatives)

labeled_pairs = pd.concat(all_parts, ignore_index=True)
labeled_pairs = labeled_pairs.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

# Summary
n_pos = (labeled_pairs['label'] == 1).sum()
n_neg = (labeled_pairs['label'] == 0).sum()

print('=' * 60)
print('📊 STAGE 8 v2 SUMMARY — Pair Construction & Label Building')
print('=' * 60)
print(f'  Total pairs   : {len(labeled_pairs):,}')
print(f'  Positive (=1) : {n_pos:,} ({n_pos/len(labeled_pairs)*100:.1f}%)')
print(f'  Negative (=0) : {n_neg:,} ({n_neg/len(labeled_pairs)*100:.1f}%)')
print(f'  Ratio neg:pos : {n_neg/max(n_pos,1):.1f}:1')
print(f'\n  Source breakdown:')
print(f'    Positive (user_folder match) : {len(positive_pairs):,}')
print(f'    Random negatives             : {len(random_negatives):,}')
print(f'    Hard negatives (candidates)  : {len(hard_negatives):,}')

# Save
labeled_pairs.to_csv(LABELED_CSV, index=False)
print(f'\n  Saved: {LABELED_CSV} ({len(labeled_pairs):,} rows)')
print(f'\n{"="*60}')
print(f'✅ Stage 8 v2 COMPLETE')
print(f'{"="*60}')

📊 STAGE 8 v2 SUMMARY — Pair Construction & Label Building
  Total pairs   : 204,701
  Positive (=1) : 29,243 (14.3%)
  Negative (=0) : 175,458 (85.7%)
  Ratio neg:pos : 6.0:1

  Source breakdown:
    Positive (user_folder match) : 29,243
    Random negatives             : 146,215
    Hard negatives (candidates)  : 29,243



  Saved: /Users/tm/Documents/GitHub/Project-for-Work/data/processed/labeled_pairs.csv (204,701 rows)

✅ Stage 8 v2 COMPLETE
